In [0]:
# Read from the Bronze Delta Lake table
df_category = (
    spark.read
    .format("delta")
    .load("abfss://bronze@mynetflixproject.dfs.core.windows.net/netflix_category")
)
display(df_category)

In [0]:
import pyspark.sql.functions as F

# 1. Check Row Counts
raw_df_category = (
    spark.read
    .option("header", "true")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("delimiter", ",")
    .csv("abfss://raw@mynetflixproject.dfs.core.windows.net/netflix_category/")
)
raw_count = raw_df_category.count()
bronze_count = df_category.count()
count_difference = bronze_count - raw_count

# 2. Standardise schemas to strings for an exact text content comparison
raw_standardised = raw_df_category.select([F.col(c).cast("string").alias(c) for c in raw_df_category.columns])
bronze_standardised = df_category.select([F.col(c).cast("string").alias(c) for c in raw_df_category.columns])

# 3. Check for specific content differences
rows_in_raw_not_bronze = raw_standardised.subtract(bronze_standardised).count()
rows_in_bronze_not_raw = bronze_standardised.subtract(raw_standardised).count()

# 4. Print Complete Validation Summary
print("=" * 45)
print("🔍 COMPLETE RAW VS. BRONZE VERIFICATION")
print("=" * 45)
print(f"Raw Row Count:                {raw_count:,}")
print(f"Bronze Row Count:             {bronze_count:,}")
print(f"Row Count Difference:         {count_difference:,}")
print("-" * 45)
print(f"Rows in Raw but NOT Bronze:   {rows_in_raw_not_bronze}")
print(f"Rows in Bronze but NOT Raw:   {rows_in_bronze_not_raw}")
print("=" * 45)

# Final Verdict Check
counts_match = (raw_count == bronze_count)
content_matches = (rows_in_raw_not_bronze == 0 and rows_in_bronze_not_raw == 0)

if counts_match and content_matches:
    print("✅ SUCCESS: Data is 100% identical in both count and content!")
else:
    print("❌ ERROR: Mismatch detected! Check the numbers above.")
print("=" * 45)


In [0]:
from pyspark.sql import functions as F

df_category = spark.read.format("delta").load(
    "abfss://bronze@mynetflixproject.dfs.core.windows.net/netflix_category"
)

# Grab the first non-key, non-metadata column name as a string using [0]
cat_col_name = [c for c in df_category.columns if c != "show_id" and not c.startswith("_")][0]
print(f"Detected category column name: '{cat_col_name}'")

cat_col = F.col(cat_col_name)
cat_trimmed = F.trim(cat_col)

# 1. Single-pass summary of quality issues
df_category.select(
    F.count("*").alias("total_rows"),
    F.countDistinct(cat_trimmed).alias("distinct_categories"),
    F.sum(F.when(cat_col.isNull() | (cat_trimmed == ""), 1).otherwise(0)).alias("dq_null_or_blank"),
    F.sum(F.when(cat_col != cat_trimmed, 1).otherwise(0)).alias("dq_outer_spaces"),
    F.sum(F.when(cat_col.rlike(r"\s{2,}"), 1).otherwise(0)).alias("dq_double_spaces"),
    F.sum(F.when(cat_col.contains(","), 1).otherwise(0)).alias("dq_contains_comma"),
    F.sum(F.when(cat_trimmed.rlike(r"^[0-9\W_]+$"), 1).otherwise(0)).alias("dq_numeric_or_symbol_only")
).display()

# 2. Print all distinct category values to spot any typos, trailing commas, or casing issues
(
    df_category
    .groupBy(cat_trimmed.alias(cat_col_name))
    .count()
    .orderBy(cat_col_name)
    .display()
)


In [0]:
from pyspark.sql import functions as F

# 1. Strip stray leading/trailing commas & spaces, collapse repeated internal spaces, blank -> NULL
staged_category_df = df_category.withColumn(
    "listed_in",
    F.when(
        F.col("listed_in").isNull() | (F.trim(F.col("listed_in")) == ""),
        F.lit(None).cast("string")
    ).otherwise(
        F.trim(
            F.regexp_replace(
                F.regexp_replace(F.col("listed_in"), r"^[,\s]+|[,\s]+$", ""),
                r"\s+",
                " "
            )
        )
    )
)

# 2. Quarantine invalid category values (NULL, blank, or pure numbers/symbols)
is_invalid_cat = (
    F.col("listed_in").isNull() |
    F.col("listed_in").rlike(r"^[0-9\W_]+$")
)

quarantine_cat_col_df = staged_category_df.filter(is_invalid_cat)
clean_category_df = staged_category_df.filter(~is_invalid_cat)

# 3. Post-transformation validation check
clean_category_df.select(
    F.count("*").alias("total_clean_rows"),
    F.countDistinct("listed_in").alias("distinct_categories"),
    F.sum(F.when(F.col("listed_in").isNull() | (F.trim(F.col("listed_in")) == ""), 1).otherwise(0)).alias("val_null_or_blank"),
    F.sum(F.when(F.col("listed_in").contains(","), 1).otherwise(0)).alias("val_remaining_commas"),
    F.sum(F.when(F.col("listed_in").rlike(r"\s{2,}"), 1).otherwise(0)).alias("val_double_spaces")
).display()


xx

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load valid parent show_ids from Silver netflix_titles
silver_titles_ids = spark.read.format("delta").load(
    "abfss://silver@mynetflixproject.dfs.core.windows.net/netflix_titles"
).select("show_id").distinct()

id_str = F.trim(F.col("show_id").cast("string"))

# 1. Single-pass audit of show_id quality AND composite duplicate (show_id, listed_in) pairs
clean_category_df.select(
    F.count("*").alias("total_rows"),
    F.sum(F.when(id_str.isNull() | (id_str == ""), 1).otherwise(0)).alias("dq_null_show_id"),
    F.sum(F.when(id_str.isNotNull() & ~id_str.rlike(r"^[0-9]+$"), 1).otherwise(0)).alias("dq_non_digit_show_id"),
    F.countDistinct(id_str).alias("distinct_show_ids"),
    (F.count("*") - F.countDistinct(id_str, F.lower(F.col("listed_in")))).alias("dq_duplicate_pairs")
).display()

# 2. Check for orphan show_ids missing from Silver netflix_titles
orphan_cat_df = (
    clean_category_df
    .withColumn("show_id_clean", id_str)
    .join(silver_titles_ids, F.col("show_id_clean") == silver_titles_ids.show_id, "left_anti")
)

print(f"Orphan show_id rows in netflix_category: {orphan_cat_df.count()}")
orphan_cat_df.display()


In [0]:
from pyspark.sql import functions as F

silver_titles_ids = spark.read.format("delta").load(
    "abfss://silver@mynetflixproject.dfs.core.windows.net/netflix_titles"
).select("show_id").distinct()

# 1. Trim whitespace and cast show_id cleanly to STRING
staged_id_df = clean_category_df.withColumn("show_id", F.trim(F.col("show_id").cast("string")))

# 2. Check foreign-key match against Silver netflix_titles
staged_with_fk = (
    staged_id_df.alias("c")
    .join(silver_titles_ids.alias("t"), F.col("c.show_id") == F.col("t.show_id"), "left")
    .select(
        F.col("c.*"),
        F.when(F.col("t.show_id").isNull(), F.lit(True)).otherwise(F.lit(False)).alias("_is_orphan")
    )
)

# 3. Quarantine rows with missing, non-digit, or orphan show_id
is_invalid_show_id = (
    F.col("show_id").isNull() |
    (F.col("show_id") == "") |
    ~F.col("show_id").rlike(r"^[0-9]+$") |
    F.col("_is_orphan")
)

quarantine_cat_id_df = (
    staged_with_fk
    .filter(is_invalid_show_id)
    .withColumn(
        "quarantine_reason",
        F.when(F.col("show_id").isNull() | (F.col("show_id") == "") | ~F.col("show_id").rlike(r"^[0-9]+$"), "Invalid or Missing show_id")
         .when(F.col("_is_orphan"), "Orphan show_id (missing in silver.netflix_titles)")
         .otherwise("Unknown show_id error")
    )
    .drop("_is_orphan")
)

# 4. Keep valid show_id rows as STRING and deduplicate composite key (show_id, listed_in)
clean_category_df = (
    staged_with_fk
    .filter(~is_invalid_show_id)
    .drop("_is_orphan")
    .dropDuplicates(["show_id", "listed_in"])
)

# 5. Validate show_id
clean_category_df.select(
    F.count("*").alias("total_clean_rows"),
    F.countDistinct("show_id").alias("distinct_show_ids"),
    F.countDistinct("show_id", "listed_in").alias("distinct_pairs"),
    F.sum(F.when(F.col("show_id").isNull() | ~F.col("show_id").rlike(r"^[0-9]+$"), 1).otherwise(0)).alias("val_invalid_show_ids")
).display()

print(f"Quarantined orphan/invalid show_id rows: {quarantine_cat_id_df.count()}")


In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

bronze_path     = "abfss://bronze@mynetflixproject.dfs.core.windows.net/netflix_category"
silver_path     = "abfss://silver@mynetflixproject.dfs.core.windows.net/netflix_category"
quarantine_path = "abfss://silver@mynetflixproject.dfs.core.windows.net/_quarantine/netflix_category"
checkpoint_path = "abfss://silver@mynetflixproject.dfs.core.windows.net/_checkpoints/netflix_category"

def process_category_microbatch(micro_df, batch_id):
    if micro_df.isEmpty():
        return

    if "_rescued_data" not in micro_df.columns:
        micro_df = micro_df.withColumn("_rescued_data", F.lit(None).cast("string"))

    silver_titles_ids = spark.read.format("delta").load(
        "abfss://silver@mynetflixproject.dfs.core.windows.net/netflix_titles"
    ).select("show_id").distinct()

    # 1. Standardise show_id (trimmed STRING) and listed_in (strip commas/spaces, collapse double spaces)
    staged = (
        micro_df
        .withColumn("show_id", F.trim(F.col("show_id").cast("string")))
        .withColumn(
            "listed_in",
            F.when(F.col("listed_in").isNull() | (F.trim(F.col("listed_in")) == ""), F.lit(None).cast("string"))
             .otherwise(
                 F.trim(
                     F.regexp_replace(
                         F.regexp_replace(F.col("listed_in"), r"^[,\s]+|[,\s]+$", ""),
                         r"\s+",
                         " "
                     )
                 )
             )
        )
    )

    # 2. Check foreign-key match against Silver netflix_titles
    staged_with_fk = (
        staged.alias("c")
        .join(silver_titles_ids.alias("t"), F.col("c.show_id") == F.col("t.show_id"), "left")
        .select(
            F.col("c.*"),
            F.when(F.col("t.show_id").isNull(), F.lit(True)).otherwise(F.lit(False)).alias("_is_orphan")
        )
    )

    # 3. Unified quarantine rules
    invalid_condition = (
        F.col("_rescued_data").isNotNull() |
        F.col("show_id").isNull() | (F.col("show_id") == "") | ~F.col("show_id").rlike(r"^[0-9]+$") |
        F.col("listed_in").isNull() | F.col("listed_in").rlike(r"^[0-9\W_]+$") |
        F.col("_is_orphan")
    )

    quarantine_df = (
        staged_with_fk.filter(invalid_condition)
        .withColumn(
            "quarantine_reason",
            F.when(F.col("_rescued_data").isNotNull(), "Corrupt CSV / Rescued Data")
             .when(F.col("show_id").isNull() | (F.col("show_id") == "") | ~F.col("show_id").rlike(r"^[0-9]+$"), "Invalid or Missing show_id")
             .when(F.col("listed_in").isNull() | F.col("listed_in").rlike(r"^[0-9\W_]+$"), "Invalid or Missing listed_in")
             .when(F.col("_is_orphan"), "Orphan show_id (missing in silver.netflix_titles)")
             .otherwise("Unknown validation failure")
        )
        .drop("_is_orphan")
        .withColumn("_quarantined_at", F.current_timestamp())
    )

    clean_df = (
        staged_with_fk.filter(~invalid_condition)
        .drop("_is_orphan", "_rescued_data")
        .dropDuplicates(["show_id", "listed_in"])
        .withColumn("_silver_processed_at", F.current_timestamp())
    )

    # 4. Write only new batch to Silver (Upsert) and Quarantine (Append)
    if not clean_df.isEmpty():
        if DeltaTable.isDeltaTable(spark, silver_path):
            (
                DeltaTable.forPath(spark, silver_path).alias("target")
                .merge(
                    clean_df.alias("source"),
                    "target.show_id = source.show_id AND target.listed_in = source.listed_in"
                )
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute()
            )
        else:
            clean_df.write.format("delta").mode("overwrite").save(silver_path)

    if not quarantine_df.isEmpty():
        quarantine_df.write.format("delta").mode("append").option("mergeSchema", "true").save(quarantine_path)

query = (
    spark.readStream
    .format("delta")
    .load(bronze_path)
    .writeStream
    .foreachBatch(process_category_microbatch)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()
print("✅ Incremental Silver processing complete for netflix_category.")


crosschech

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

ACCOUNT = "mynetflixproject"

def lake(container: str, *parts: str) -> str:
    clean_parts = [p.strip("/") for p in parts if p and p.strip("/")]
    return f"abfss://{container}@{ACCOUNT}.dfs.core.windows.net/{'/'.join(clean_parts)}"

# 1. Load parent table (netflix_titles) and verify primary key uniqueness
titles_df = spark.read.format("delta").load(lake("silver", "netflix_titles"))
titles_ids = titles_df.select("show_id").distinct()

titles_cnt = titles_df.count()
titles_distinct_cnt = titles_ids.count()
assert titles_cnt == titles_distinct_cnt, f"❌ Primary key violation in netflix_titles: {titles_cnt} rows vs {titles_distinct_cnt} distinct IDs!"

# 2. Define lookup tables and their composite natural keys
lookup_specs = {
    "netflix_directors": "director",
    "netflix_cast": "cast",
    "netflix_countries": "country",
    "netflix_category": "listed_in"
}

summary_rows = [
    ("netflix_titles", titles_cnt, titles_distinct_cnt, 0,
     spark.read.format("delta").load(lake("silver", "_quarantine", "netflix_titles")).count()
     if DeltaTable.isDeltaTable(spark, lake("silver", "_quarantine", "netflix_titles")) else 0)
]

# 3. Audit each lookup table for schema types, duplicate pairs, and orphan foreign keys
for tbl, val_col in lookup_specs.items():
    df = spark.read.format("delta").load(lake("silver", tbl))
    q_path = lake("silver", "_quarantine", tbl)
    q_cnt = spark.read.format("delta").load(q_path).count() if DeltaTable.isDeltaTable(spark, q_path) else 0
    
    total_cnt = df.count()
    distinct_pairs = df.select("show_id", val_col).distinct().count()
    orphan_cnt = df.join(titles_ids, on="show_id", how="left_anti").count()
    
    # Hard assertions
    assert dict(df.dtypes)["show_id"] == "string", f"❌ {tbl}.show_id is not STRING!"
    assert total_cnt == distinct_pairs, f"❌ Duplicate pairs found in {tbl}!"
    assert orphan_cnt == 0, f"❌ {orphan_cnt} orphan show_ids found in {tbl}!"
    
    summary_rows.append((tbl, total_cnt, distinct_pairs, orphan_cnt, q_cnt))

# 4. Display unified Silver scorecard
scorecard_df = spark.createDataFrame(
    summary_rows,
    ["table_name", "clean_silver_rows", "distinct_natural_keys", "orphan_show_ids", "quarantined_rows"]
)

print("✅ ALL 5 SILVER TABLES PASSED SCHEMA, UNIQUENESS, AND REFERENTIAL INTEGRITY GATES!")
scorecard_df.display()
